# Analisi dati per effetto Hall

A grandi linee:
1. caratterizzare l'**uniformità del campo magnetico** nel traferro
2. caratterizzare l'andamento del **campo magnetico** nel traferro al **variare della corrente** in ingresso
3. caratterizzare **tensione di Hall al variare della corrente** $V_H(i)$ (ferromagnete spento + 5x2 valori del campo magnetico)
4. caratterizzare **tensione di Hall al variare del campo magnetico** $V_H(B)$ (ferromagnete spento + 3x2 valori di corrente)
5. valutare **mobilità** dei portatori di carica nel materiale

+ cose facolative (?)

### utils

In [1]:
import numpy as np
from utils import meanCalc, fitPlotter, testZ, MeanError, Zscore, Amprobe, Keithley, Teslameter
import ROOT

In module 'Darwin':
/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX.sdk/usr/include/xlocale/_ctype.h:55:1: error: '__toupper_l' has different definitions in different modules; definition in module 'Darwin.C.xlocale._ctype' first difference is function body
__toupper_l(__darwin_ct_rune_t _c, locale_t _l)
^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX.sdk/usr/include/xlocale/_ctype.h:56:2: note: but in '_DarwinFoundation2._ctype.xlocale' found a different body
{
~^
In module 'Darwin':
/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX.sdk/usr/include/xlocale/_ctype.h:62:1: error: '__tolower_l' has different definitions in different modules; definition in module 'Darwin.C.xlocale._ctype' first difference is function body
__tolower_l(__darwin_ct_rune_t _c, locale_t _l)
^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
/

## Uniformità campo magnetico (TOTHINKABOUT)

Facendo misure del campo magnetico con una sonda di Hall vogliamo studiare l'uniformità di $B$ all'interno del traferro del ferromagnete.

In generale, la semidifferenza tra il valore massimo e il valore minimo del campo magnetico (all'interno della regione di omogeneità) sarà il limite minimo dell'errore su tutte le misure di campo magnetico.

In [4]:
# arrays of distances
posx = np.arange(5)*10
posy = np.arange(5)*10

# matrix of B values in mT
Bu = np.array([[176.20,204.5,202.0,203.6,164.16],
               [205.98,234.51,233.37,234.15,211.61],
               [215.00,236.,236.47,236.65,209.43],
               [204.67,235.5,235.5,235.3,207.4],
               [176.24,200.,201.10,202.05,175.02]])

Can2d_u = ROOT.TCanvas("c2d", "Uniformità campo magnetico")
H2d_u = ROOT.TH2F("h2d_u", "Uniformità campo magnetico; x [mm]; y [mm]; B [T]" ,5,0,5, 5,0,5)

for i in range(5):
    for j in range(5):
        H2d_u.SetBinContent(i+1,j+1,float(Bu[i][j]))

H2d_u.Draw("LEGO2")
Can2d_u.Draw()

Dopo lo studio di uniformità del campo magnetico, si stabilisce in quale zona effettuare le misure di campo magnetico. In tale zona, si prende la semidispersione di B come stima della risoluzione del campo magnetico.

In [5]:
# we calculate the semidifference to get an estimate on the error on the measurement of B

B_max = 236.65
B_min = 233.4
resB = (B_max - B_min)/2
print(resB) 

1.625


## Campo magnetico prodotto al variare della corrente

Tralasciando la prima curva di salita (da $0$ a $i_\text{max}$) misuriamo due salite e due discese complete (da $i_\text{max}$ a $-i_\text{max}$, e viceversa) prendendo 10 dati per ogni curva. Sostanzialmente, vogliamo verificare la regione di linearità in cui (dopo) vogliamo svolgere il resto dell'esperienza.

Escluse le zone di saturazione vogliamo un fare un fit lineare su ogni curva: per ognuna delle due coppie (due curve di salita, e due curve di discesa) stimiamo coi valori medi di $m$ e $q$ il vero coefficiente angolare e la vera quota (*l'errore sulla quota lo stimiamo con la semidifferenza tra i due valori*).
Solo dopo mediamo i risultati ottenuti per le curve di salita e quelle di discesa (*errore sulla quota sempre dato dalla semidifferenza*).

In [6]:
# we would like to see the hystheresis loop
# answer: i believe that the greatest contribution is given by resB (see cell above), however we need to check with the teslamete

# first negative run

B1 = np.array([355.2, 322.5, 279.3, 243, 200.4, 159, 104.5, 59.8, 9.3, -47.7, -87.7, -136.4, -183.3, -228.5, -269.9, -314.1, -349.4, -383.8]) #mT
i1 = np.array([1.602, 1.406, 1.18, 0.999, 0.8, 0.619, 0.393, 0.208, 0, -0.234, -0.4, -0.603, -0.802, -1.011, -1.201, -1.419, -1.604, -1.805]) #A
errB1 = Teslameter(B1)
erri1 = Amprobe(i1, unit = "A")

# first positive run
B2 = np.array([-358.1, -322.4, -282.9, -244.4, -201.6, -154.4, -94.7, -58.9, -8.7, 40.2, 89.6, 138.9, 189.1, 231.9, 268.8, 310.6, 350.7, 381.7]) #mT
i2 = np.array([-1.605, -1.4, -1.195, -1.007, -0.806, -0.599, -0.353, -0.206, 0, 0.202, 0.405, 0.611, 0.825, 1.025, 1.198, 1.403, 1.615, 1.798]) #A
errB2 = Teslameter(B2)
erri2 = Amprobe(i2,unit = "A")

# second negative run
B3 = np.array([356.3, 319.6, 281.7, 243.8, 198, 157.7, 108.3, 56.7, 8.9, -44.5, -88.4, -141.8, -185.5, -227.3, -272.3, -313.6, -349.9, -383.8]) #mT
i3 = np.array([1.599, 1.383, 1.192, 1.003, 0.788, 0.611, 0.409, 0.197, 0, -0.219, -0.4, -0.621, -0.808, -1.001, -1.213, -1.416, -1.606, -1.807]) #A
errB3 = Teslameter(B3)
erri3 = Amprobe(i3, unit = "A") 

# second positive run 
# we have five more data as we wanted to see the magnetic saturation
B4 = np.array([-358.7, -325.1, -284.8, -240.8, -202.5, -156, -106.2, -56.4, -8.7, 42.5, 87.2, 137.5, 190.2, 227.7, 272.7, 321.9, 349.5, 381.7, 414.9, 452, 476.7, 495.4, 499.9]) #mT
i4 = np.array([-1.61, -1.409, -1.201, -0.987, -0.808, -0.603, -0.399, -0.195, 0, 0.21, 0.395, 0.602, 0.83, 1, 1.208, 1.456, 1.602, 1.798, 2.013, 2.318, 2.627, 2.9, 2.977]) #A
errB4 = Teslameter(B4)
erri4 = Amprobe(i4, unit = "A")

# here we fix the minimum uncertainty on the magnetic fields using the semidifference
errB1[errB1 < resB] = resB
errB2[errB2 < resB] = resB
errB3[errB3 < resB] = resB
errB3[errB3 < resB] = resB

# we have discarded some data as they were difficult to fit
magCurrPlotter = fitPlotter("MagneticFieldVsCurrent")
param1 = magCurrPlotter.addGraph(i1[2:-3], B1[2:-3], erri1[2:-3], errB1[2:-3], title="Run Down 1")
param2 = magCurrPlotter.addGraph(i2[2:-3], B2[2:-3], erri2[2:-3], errB2[2:-3], title="Run Up 1")
param3 = magCurrPlotter.addGraph(i3[2:-3], B3[2:-3], erri3[2:-3], errB3[2:-3], title="Run Down 2")
param4 = magCurrPlotter.addGraph(i4[2:-8], B4[2:-8], erri4[2:-8], errB4[2:-8], title="Run Up 2 (longer)")
magCurrPlotter.drawCanvas()
magCurrPlotter.saveCanvas("outputs/MagneticFieldVsCurrent.png")


--- fit Results for: Run Down 1 ---
Function: pol1
Chi2/NDF: 6.5421 / 11
p-value:  0.8349

p0: 9.4118 +/- 0.7618
p1: 238.2000 +/- 1.9343
--------------------------------

--- fit Results for: Run Up 1 ---
Function: pol1
Chi2/NDF: 6.7369 / 11
p-value:  0.8200

p0: -9.0841 +/- 0.7604
p1: 238.0598 +/- 1.9833
--------------------------------

--- fit Results for: Run Down 2 ---
Function: pol1
Chi2/NDF: 6.3637 / 11
p-value:  0.8480

p0: 8.7286 +/- 0.8243
p1: 237.8032 +/- 2.0921
--------------------------------

--- fit Results for: Run Up 2 (longer) ---
Function: pol1
Chi2/NDF: 8.1593 / 11
p-value:  0.6990

p0: -8.7478 +/- 0.5579
p1: 240.2872 +/- 1.6928
--------------------------------


python ERROR: cannot open image file "outputs/MagneticFieldVsCurrent.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/MagneticFieldVsCurrent.png


NB: per i parametri ritornati dai fit, sto utilizzando il formato `np.array([[mean][error],...])` per scrivere i valori, così poi da avere le funzioni Zscore e MeanErrors per calcolare direttamente valori attessi ed errori

In [7]:
# verifico che i parametri in salita e discesa siano compatibili, poi medio tutto

# BE ATTENTIVE: we have to take the error as the semidifference?

Zscore(param1, param3)

param_down = MeanError(param1, param3)

Zscore(param2, param4)
param_up = MeanError(param2, param4)

Zscore(param_up, param_down)
param = MeanError(param_up,param_down)

param, param_up, param_down

z value of param 0 : -0.609
z value of param 1 : -0.139


z value of param 0 : 0.357
z value of param 1 : 0.854


z value of param 0 : 12.258
z value of param 1 : -0.303




(array([[7.71592958e-02, 1.25711581e-02],
        [2.38587536e+02, 3.86467521e+00]]),
 array([[ -8.91592049,  -0.93830804],
        [239.17350565,   2.6095247 ]]),
 array([[  9.07023908,   1.12810928],
        [238.00156685,   2.84945459]]))

In [8]:
# here we need to build the functions to translate current that goes in the electromagnet into magnetic field
# be mindful of the fact that we are using the named parameters in the cell above as default values

def hysteresisUp(I, err_I, q=param_up[0][0], err_q=param_up[0][1], m=param_up[1][0], err_m=param_up[1][1]):
    """ returns  B, errB (mag field) np.arrays after taking I, errI (current), for hystUP. optionally hysteresis curve parameters """
    B    = q + m * I
    errB = np.sqrt(err_q**2 + (I*err_m)**2 + (m*err_I)**2)

    return B, errB

def hysteresisDown(I, err_I, q=param_down[0][0], err_q=param_down[0][1], m=param_down[1][0], err_m=param_down[1][1]):
    """ returns  B, errB (mag field) np.arrays after taking I, errI (current), for hystDOWN. optionally hysteresis curve parameters """
    B    = q + m * I
    errB = np.sqrt(err_q**2 + (I*err_m)**2 + (m*err_I)**2)

    return B, errB

def hysteresisMean(I, err_I, q=param[0][0], err_q=param[0][1], m=param[1][0], err_m=param[1][1]):
    """ returns  B, errB (mag field) np.arrays after taking I, errI (current). optionally hysteresis curve parameters """
    B    = q + m * I
    errB = np.sqrt(err_q**2 + (I*err_m)**2 + (m*err_I)**2)

    return B, errB

## Tensione di Hall al variare di $i$

Vogliamo valutare l'andamento della tensione di Hall (sui lati del materiale semiconduttore) al variare della corrente iniettata al suo interno, in modo da stimare il parametro $R_H$.

Quindi fissato il valore del campo magnetico (una volta a zero, e poi a 5 valori diversi in entrambi i versi) valutiamo l'andamento $V_H(i)$ (*andando tra **-8mA e +8mA***) sapendo che in generale:
$$V_H = \frac{R_H}{t} \cdot i_p \cdot B$$
allora facciamo dei fit lineari su ogni set di dati e diamo una stima del parametro $R_H$ per entrambi i versi del campo magnetico, poi mediamo tra i due set.

In [18]:
# correction for misalignment of transverse contacts
# we will need to subtract it from the other fits

VH0    = np.array([-0.001, 0.055, 0.109, 0.165, 0.22, 0.276, 0.328, 0.384, 0.439, -0.051, -0.104, -0.159, -0.214, -0.267, -0.319, -0.374, -0.429])
errVH0 = Keithley(VH0)
Ip0    = np.array([0, 1.009, 1.999, 2.989, 4, 5.012, 5.977, 6.977, 7.997, -0.996, -1.986, -2.998, -4.003, -4.995, -5.975, -6.985, -8.009])
errIp0 = Amprobe(Ip0,unit="mA")


hallCurrPlotter = fitPlotter("HallTensionVsCurrent")
param0 = hallCurrPlotter.addGraph(Ip0, VH0, errIp0, errVH0, "Ohmic behaviour")
hallCurrPlotter.drawCanvas()
hallCurrPlotter.saveCanvas("outputs/HallTensionVsCurrent.png")


--- fit Results for: Ohmic behaviour ---
Function: pol1
Chi2/NDF: 3.9827 / 15
p-value:  0.9978

p0: 0.0032 +/- 0.0009
p1: 0.0542 +/- 0.0002
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallTensionVsCurrent
python ERROR: cannot open image file "outputs/HallTensionVsCurrent.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/HallTensionVsCurrent.png


In [24]:
# specifically we define the ohmic behaviour function to make corrections afterward (to the Vh fits' results)

def ohmicContribution(I, errI,q=param0[0][0],errq=param0[0][1],m=param0[1][0],errm=param0[1][1]):
    """ returns Vh(I), errVh(I) for B=0 """
    Vh = q + m * I
    errVh = np.sqrt(errq**2 + (I*errm)**2 + (m*errI)**2)

    return Vh, errVh

# these are the corrections we need to apply!
I = np.array([-2.002, -4.003, -6.005, 2.008, 4.000, 6.000])
errI = Amprobe(I, unit="mA")
ohmicContribution(I, errI)

(array([-0.10532297, -0.21377062, -0.32227247,  0.11200591,  0.21996579,
         0.32835925]),
 array([0.00129457, 0.00182589, 0.0024295 , 0.00129596, 0.00182502,
        0.00242794]))

In [10]:
# we are fitting VH(I), using Amprobe and Keithley function to calculate the error

i1 = np.array([0, -0.995, -1.973, -2.989, -3.994, -4.988, -5.972, -7.002, -7.985, 1.001, 2.001, 3.019, 4, 4.993, 5.989, 7.002, 8.014]) #mA
VH1 = np.array([-0.01, -0.479, -0.938, -1.405, -1.875, -2.341, -2.801, -3.284, -3.743, 0.461, 0.93, 1.406, 1.866, 2.331, 2.798, 3.273, 3.747]) #mV
erri1 = Amprobe(i1, unit = "mA")
errVH1 = Keithley(VH1)

i2 = np.array([0, 0.988, 1.996, 2.989, 4, 4.998, 6.004, 6.959, 8.012, -1.011, -1.968, -3.005, -4.005, -5.052, -5.98, -7.033, -8.004]) #mA
VH2 = np.array([-0.008, 0.847, 1.721, 2.58, 3.455, 4.319, 5.189, 6.016, 6.927, -0.885, -1.714, -2.612, -3.477, -4.383, -5.185, -6.096, -6.936]) #mV
erri2 = Amprobe(i2, unit = "mA")
errVH2 = Keithley(VH2)

i3 = np.array([0, -1.001, -2.011, -3.029, -4.043, -4.983, -5.987, -6.999, -8.037, 1.002, 2.02, 2.995, 4.043, 5.003, 6.045, 7.012, 7.993]) #mA
VH3 = np.array([-0.009, -1.227, -2.452, -3.689, -4.919, -6.06, -7.279, -8.506, -9.765, 1.21, 2.445, 3.629, 4.899, 6.065, 7.329, 8.501, 9.691]) #mV
erri3 = Amprobe(i3, unit = "mA")
errVH3 = Keithley(VH3)


i4 = np.array([0, 1.025, 2.002, 3.038, 4.058, 5.032, 6.012, 7, 8.006, -1.002, -2.002, -2.998, -4.034, -4.992, -6.013, -7.015, -7.987]) #mA
VH4 = np.array([-0.008, 1.592, 3.113, 4.728, 6.317, 7.833, 9.358, 10.896, 12.461, -1.57, -3.127, -4.678, -6.291, -7.781, -9.368, -10.926, -12.436]) #mV
erri4 = Amprobe(i4, unit = "mA") 
errVH4 = Keithley(VH4)

i5 = np.array([0, -1.002, -2.002, -3.006, -3.992, -5.02, -6.003, -7.036, -8.021, 1.008, 1.988, 3, 3.997, 5.015, 5.962, 7.018, 8]) #mA
VH5 = np.array([-0.007, -1.891, -3.768, -5.653, -7.5, -9.415, -11.255, -13.189, -15.031, 1.884, 3.719, 5.614, 7.477, 9.381, 11.151, 13.127, 14.96]) #mV
erri5 = Amprobe(i5, unit = "mA")
errVH5 = Keithley(VH5)

hallVoltagePlotter = fitPlotter("HallVoltageVsCurrentBneg")

# from these parameters we need to remove the ohmic contribution
param1neg = hallVoltagePlotter.addGraph(i1,VH1,erri1,errVH1, title="i magnet -0.2 A")
param2neg = hallVoltagePlotter.addGraph(i2,VH2,erri2,errVH2, title="i magnet -0.4 A")
param3neg = hallVoltagePlotter.addGraph(i3,VH3,erri3,errVH3, title="i magnet -0.6 A")
param4neg = hallVoltagePlotter.addGraph(i4,VH4,erri4,errVH4, title="i magnet -0.8 A")
param5neg = hallVoltagePlotter.addGraph(i5,VH5,erri5,errVH5, title="i magnet -1.0 A")

hallVoltagePlotter.drawCanvas()
hallVoltagePlotter.saveCanvas("outputs/hallVoltageVsCurrentBneg.png")


--- fit Results for: i magnet -0.2 A ---
Function: pol1
Chi2/NDF: 1.7161 / 15
p-value:  1.0000

p0: -0.0091 +/- 0.0021
p1: 0.4683 +/- 0.0008
--------------------------------

--- fit Results for: i magnet -0.4 A ---
Function: pol1
Chi2/NDF: 0.0760 / 15
p-value:  1.0000

p0: -0.0083 +/- 0.0027
p1: 0.8659 +/- 0.0014
--------------------------------

--- fit Results for: i magnet -0.6 A ---
Function: pol1
Chi2/NDF: 0.1175 / 15
p-value:  1.0000

p0: -0.0091 +/- 0.0030
p1: 1.2143 +/- 0.0020
--------------------------------

--- fit Results for: i magnet -0.8 A ---
Function: pol1
Chi2/NDF: 0.1560 / 15
p-value:  1.0000

p0: -0.0077 +/- 0.0032
p1: 1.5577 +/- 0.0025
--------------------------------

--- fit Results for: i magnet -1.0 A ---
Function: pol1
Chi2/NDF: 0.6287 / 15
p-value:  1.0000

p0: -0.0076 +/- 0.0033
p1: 1.8738 +/- 0.0030
--------------------------------


python ERROR: cannot open image file "outputs/hallVoltageVsCurrentBneg.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsCurrentBneg.png


In [11]:
# same mesaures but we inverted the magnetic field
# we still need to implement the errors on Hall tension
# we are fitting VH(I), using Amprobe and Keithley function to calculate the error

i1 = np.array([0, 1.007, 1.988, 2.991, 4.011, 5, 5.988, 7.009, 8.001, -1.002, -1.999, -3.006, -3.997, -5.039, -6.014, -6.953, -8.07])
VH1 = np.array([-0.002, -0.267, -0.525, -0.788, -1.056, -1.316, -1.575, -1.843, -2.104, 0.261, 0.522, 0.786, 1.046, 1.32, 1.576, 1.822, 2.115])
erri1 = Amprobe(i1, unit = "mA")
errVH1 = Keithley(VH1)

i2 = np.array([0, -1.025, -1.902, -2.997, -4.049, -4.955, -6.075, -7.068, -7.974, 0.922, 2.003, 3.02, 4.006, 5.021, 5.953, 6.998, 7.978])
VH2 = np.array([-0.002, 0.684, 1.267, 2.001, 2.704, 3.309, 4.057, 4.719, 5.325, -0.618, -1.345, -2.027, -2.688, -3.369, -3.995, -4.696, -5.355])
erri2 = Amprobe(i2, unit = "mA")
errVH2 = Keithley(VH2)

i3 = np.array([0, 1.011, 1.934, 2.984, 3.94, 4.944, 5.988, 7.042, 8.029, -0.985, -1.912, -3.007, -4.036, -4.945, -5.925, -6.98, -7.992])
VH3 = np.array([-0.001, -1.056, -2.021, -3.114, -4.112, -5.159, -6.248, -7.347, -8.377, 1.028, 1.997, 3.139, 4.211, 5.16, 6.183, 7.283, 8.338])
erri3 = Amprobe(i3, unit = "mA")
errVH3 = Keithley(VH3)

i4 = np.array([0, -1.006, -2.016, -2.972, -3.973, -5.01, -5.966, -6.996, -7.97, 0.975, 1.993, 2.997, 4.001, 4.983, 6.004, 6.978, 7.984])
VH4 = np.array([0, 1.419, 2.838, 4.186, 5.594, 7.053, 8.398, 9.846, 11.216, -1.372, -2.806, -4.217, -5.268, -7.011, -8.446, -9.814, -11.227])
erri4 = Amprobe(i4, unit = "mA")
errVH4 = Keithley(VH4)

i5 = np.array([0, 0.97, 1.998, 2.983, 4.002, 5.001, 6.009, 6.997, 7.971, -0.997, -2.007, -2.996, -3.987, -5.011, -6, -7.007, -8.013])
VH5 = np.array([0, -1.665, -3.415, -5.122, -6.867, -8.58, -10.308, -12.001, -13.668, 1.713, 3.443, 5.139, 6.847, 8.586, 10.28, 12.003, 13.728])
erri5 = Amprobe(i4, unit = "mA")
errVH5 = Keithley(VH5)

hallVoltagePlotter = fitPlotter("HallVoltageVsCurrentBpos")

# from these parameters we need to remove the ohmic contribution
param1pos = hallVoltagePlotter.addGraph(i1,VH1,erri1,errVH1, title="i magnet 0.2 A")
param2pos = hallVoltagePlotter.addGraph(i2,VH2,erri2,errVH2, title="i magnet 0.4 A")
param3pos = hallVoltagePlotter.addGraph(i3,VH3,erri3,errVH3, title="i magnet 0.6 A")
param4pos = hallVoltagePlotter.addGraph(i4,VH4,erri4,errVH4, title="i magnet 0.8 A")
param5pos = hallVoltagePlotter.addGraph(i5,VH5,erri5,errVH5, title="i magnet 1.0 A")

hallVoltagePlotter.drawCanvas()
hallVoltagePlotter.saveCanvas("outputs/hallVoltageVsCurrentBpos.png")


--- fit Results for: i magnet 0.2 A ---
Function: pol1
Chi2/NDF: 0.1025 / 15
p-value:  1.0000

p0: -0.0027 +/- 0.0015
p1: -0.2625 +/- 0.0005
--------------------------------

--- fit Results for: i magnet 0.4 A ---
Function: pol1
Chi2/NDF: 1.1588 / 15
p-value:  1.0000

p0: -0.0033 +/- 0.0025
p1: -0.6694 +/- 0.0011
--------------------------------

--- fit Results for: i magnet 0.6 A ---
Function: pol1
Chi2/NDF: 0.0710 / 15
p-value:  1.0000

p0: -0.0008 +/- 0.0029
p1: -1.0436 +/- 0.0017
--------------------------------

--- fit Results for: i magnet 0.8 A ---
Function: pol1
Chi2/NDF: 98.1696 / 15
p-value:  0.0000

p0: 0.0031 +/- 0.0031
p1: -1.4018 +/- 0.0023
--------------------------------

--- fit Results for: i magnet 1.0 A ---
Function: pol1
Chi2/NDF: 0.4492 / 15
p-value:  1.0000

p0: 0.0001 +/- 0.0032
p1: -1.7148 +/- 0.0028
--------------------------------


python ERROR: cannot open image file "outputs/hallVoltageVsCurrentBpos.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsCurrentBpos.png


In [12]:
# here the parameters are stored as [[q, qerr], [m, merr]]
# sto sottraendo i parametri della caratteristica V_H(i) con B=0 a V_H(i) a B variabile

parampos = [param1pos, param2pos, param3pos, param4pos, param5pos]
paramneg = [param1neg, param2neg, param3neg, param4neg, param5neg]

m     = []
err_m = []

q     = []
err_q = []

for pos in parampos:
    q.append(pos[0][0] - param0[0][0])
    err_q.append(np.sqrt(pos[0][1]**2 + param0[0][1]**2))

    m.append(pos[1][0] - param0[1][0])
    err_m.append(np.sqrt(pos[1][1]**2 + param0[1][1]**2))

for pos in paramneg:
    q.append(pos[0][0] - param0[0][0])
    err_q.append(np.sqrt(pos[0][1]**2 + param0[0][1]**2))

    m.append(pos[1][0] - param0[1][0])
    err_m.append(np.sqrt(pos[1][1]**2 + param0[1][1]**2))

m = np.array(m)
err_m = np.array(err_m)

q = np.array(q)
err_q = np.array(err_q)

In [13]:
# HERE WE PERFORM LINEAR FIT of M (slope of V_H(i) curves against B) to get R_H!

Ineg = np.array([-0.200,-0.425,-0.616,-0.811, -1.005])
errIneg = Amprobe(Ineg, unit="A")

Ipos = np.array([0.200, 0.409, 0.606, 0.813, 0.999])
errIpos = Amprobe(Ipos, unit="A")

Bneg, errBneg = hysteresisMean(Ineg, errIneg)
Bpos, errBpos = hysteresisMean(Ipos, errIpos)

# this is the order in which i get the slopes: parampos then paramneg
Bposneg    = np.concatenate((Bpos, Bneg), axis=None)
errBposneg = np.concatenate((errBpos, errBneg), axis=None)

RHfromVhVsIplotter = fitPlotter("SlopesVhIVsB")

# where coeff = R_H / t
[stuff, [coeff, err_coeff]] = RHfromVhVsIplotter.addGraph(m, Bposneg, err_m, errBposneg, title="Slopes against B")
RHfromVhVsIplotter.drawCanvas()
RHfromVhVsIplotter.saveCanvas("outputs/slopesAgainstBtogetRH.png")

# here we determine R_H
t = 1
err_t = 0.1 # need to check: it's a big error...

# IN WHICH UNITS?
R_H    = coeff * t
errR_H = np.sqrt((coeff*err_t)**2 + (t*err_coeff)**2)


--- fit Results for: Slopes against B ---
Function: pol1
Chi2/NDF: 3.6038 / 8
p-value:  0.8910

p0: 6.1865 +/- 0.5530
p1: -130.7689 +/- 1.1126
--------------------------------


python ERROR: cannot open image file "outputs/slopesAgainstBtogetRH.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/slopesAgainstBtogetRH.png


## Tensione di Hall al variare di $B$

Ripetiamo sostanzialmente le misure al punto precedente, ma invertendo i ruoli di variabile dipendente e indipendente. Adesso fissiamo $i_p$, scegliendo $3 \times 2$ valori, (dopo aver tracciato un'altra curva di caduta di potenziale ohmica a $B=0$) e studiamo il variare di $V_H$ con $B$.

**probabilmente ha senso riprendere i valori della curva ohmica (disallineamento) fissando i valori della corrente in base a questa nuova scansione: altrimenti dobbiamo interpolare lungo la curva di un fit**

In [17]:
# WARNING the arrays IB1,IB2 etc... contain current measurements!!! 
# same thing as before but now we are changing the magnetic field
# we still need to find the error on B

# here error propagation on currents and tensions with multimeters

IB1 = np.array([-1.195, -0.88, -0.602, -0.287, 0, 0.41, 0.65, 0.902, 1.178])
errIB1 = Amprobe(IB1, unit="A")
VH1 = np.array([4.319, 3.483, 2.554, 1.371, 0.236, -1.376, -2.276, -3.148, -4.014])
errVH1 = Keithley(VH1)

IB2 = np.array([1.177, 0.711, 0.552, 0.304, 0, -0.315, -0.597, -0.978, -1.238])
errIB2 = Amprobe(IB2, unit="A")
VH2 = np.array([-7.997, -5.371, -4.271, -2.413, -0.018, 2.468, 4.605, 7.252, 8.844])
errVH2 = Keithley(VH2)

IB3 = np.array([-1.237, -0.894, -0.609, -0.334, 0, 0.342, 0.587, 0.915, 1.189])
errIB3 = Amprobe(IB3, unit="A")
VH3 = np.array([13.258, 10.537, 7.704, 4.642, 0.705, -3.335, -6.106, -9.544, -12.09])
errVH3 = Keithley(VH3)

# now we translate currents in electromagnet to mag fields with specific hysteresis

B1, errB1 = hysteresisUp(IB1, errIB1)
B2, errB2 = hysteresisUp(IB2, errIB2)
B3, errB3 = hysteresisUp(IB3, errIB3)

hallVoltageBFieldPlotter = fitPlotter("HallVoltageVsMagneticFieldIpos")

param1 = hallVoltageBFieldPlotter.addGraph(B1,VH1,errB1,errVH1, title="i sensor 0.2 A")
param2 = hallVoltageBFieldPlotter.addGraph(B2,VH2,errB2,errVH2, title="i sensor 0.4 A")
param3 = hallVoltageBFieldPlotter.addGraph(B3,VH3,errB3,errVH3, title="i sensor 0.6 A")

hallVoltageBFieldPlotter.drawCanvas()
hallVoltageBFieldPlotter.saveCanvas("outputs/hallVoltageVsBFieldIpos.png")


--- fit Results for: i sensor 0.2 A ---
Function: pol1
Chi2/NDF: 24.1060 / 7
p-value:  0.0011

p0: 0.1087 +/- 0.0122
p1: -0.0157 +/- 0.0002
--------------------------------

--- fit Results for: i sensor 0.4 A ---
Function: pol1
Chi2/NDF: 32.4749 / 7
p-value:  0.0000

p0: -0.2977 +/- 0.0203
p1: -0.0318 +/- 0.0003
--------------------------------

--- fit Results for: i sensor 0.6 A ---
Function: pol1
Chi2/NDF: 33.7645 / 7
p-value:  0.0000

p0: 0.2697 +/- 0.0310
p1: -0.0476 +/- 0.0004
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallVoltageVsMagneticFieldIpos
python ERROR: cannot open image file "outputs/hallVoltageVsBFieldIpos.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsBFieldIpos.png


In [15]:
# now the current flows the other way

IB1 = np.array([1.207, 0.9, 0.546, 0.277, 0, -0.3, -0.497, -0.918, -1.211])
errIB1 = Amprobe(IB1, unit="A")
VH1 = np.array([4.093, 3.307, 2.132, 1.118, 0.025, -1.16, -1.918, -3.426, -4.348])
errVH1 = Keithley(VH1)

IB2 = np.array([-1.211, -0.907, -0.602, -0.312, 0, 0.303, 0.725, 0.897, 1.225])
errIB2 = Amprobe(IB2, unit="A")
VH2 = np.array([-8.697, -7.102, -5.088, -2.919, -0.462, 1.935, 5.078, 6.252, 8.284])
errVH2 = Keithley(VH2)

IB3 = np.array([1.221, 0.879, 0.604, 0.321, 0, -0.4, -0.638, -0.928, -1.197])
errIB3 = Amprobe(IB3, unit="A")
VH3 = np.array([12.403, 9.723, 6.992, 3.838, 0.048, -4.667, -7.339, -10.372, -12.901])
errVH3 = Keithley(VH3)

# now we translate currents in electromagnet to mag fields with specific hysteresis

B1, errB1 = hysteresisUp(IB1, errIB1)
B2, errB2 = hysteresisUp(IB2, errIB2)
B3, errB3 = hysteresisUp(IB3, errIB3)

hallVoltageBFieldPlotter = fitPlotter("HallVoltageVsMagneticFieldNeg")

param1 = hallVoltageBFieldPlotter.addGraph(B1,VH1,errB1,errVH1,title="i sensor -0.2 A")
param2 = hallVoltageBFieldPlotter.addGraph(B2,VH2,errB2,errVH2,title="i sensor -0.4 A")
param3 = hallVoltageBFieldPlotter.addGraph(B3,VH3,errB3,errVH3,title="i sensor -0.6 A")

hallVoltageBFieldPlotter.drawCanvas()
hallVoltageBFieldPlotter.saveCanvas("outputs/hallVoltageVsBFieldNeg.png")


--- fit Results for: i sensor -0.2 A ---
Function: pol1
Chi2/NDF: 34.3569 / 7
p-value:  0.0000

p0: 0.1625 +/- 0.0101
p1: 0.0159 +/- 0.0001
--------------------------------

--- fit Results for: i sensor -0.4 A ---
Function: pol1
Chi2/NDF: 33.5949 / 7
p-value:  0.0000

p0: -0.1723 +/- 0.0203
p1: 0.0317 +/- 0.0003
--------------------------------

--- fit Results for: i sensor -0.6 A ---
Function: pol1
Chi2/NDF: 27.5946 / 7
p-value:  0.0003

p0: 0.5023 +/- 0.0342
p1: 0.0471 +/- 0.0005
--------------------------------


python ERROR: cannot open image file "outputs/hallVoltageVsBFieldNeg.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsBFieldNeg.png


## Mobilità dei portatori

Misurando la caratteristica $I(V)$ del materiale seminconduttore (facendo variare la corrente $I$ tra -8mA e +8mA) che abbiamo usato nell'esperienza possiamo dare una stima della mobilità dei portatori di carica al suo interno. In particolare, dalla pendenza di $I(V)$ abbiamo la resistenza $R$, e quindi anche la resistività:
$$\rho =\frac{t\cdot w}{L} R$$
dove $t$ è lo spessore, $w$ la larghezza e $L$ la lunghezza.
Infine, dalla resistività si ha direttamente:
$$\mu = \frac{R_H}{\rho} (=R_H \sigma)$$
dove per $R_H$ possiamo considerare le stime date prima.

In [ ]:
# fitting the curve I(V) as we need to find R in order tu calculate mu
# We miss 0 point

I   = np.array([-0.501, -1.01, -1.503, -1.999, -2.492, -3, -3.499, -4.004, -4.494, -5.002, -5.504, -6.018, -6.5, -6.998, -7.505, -8.002,0.537, 1.018, 1.514, 1.987, 2.499, 2.996, 3.498, 4.01, 4.49, 5.016, 5.451, 5.973, 6.47, 7, 7.5, 8]) #mA
errI = Amprobe(I, unit = "mA")
V    = [-32.505, -64.937, -97.343, -129.5, -161.399, -194.292, -226.477, -259.082, -290.841, -323.673, -356.178, -389.277, -420.475, -452.676, -484.506, -517.489,34.804, 65.991, 98.036, 128.64, 161.732, 193.9, 226.29, 259.353, 290.42, 324.444, 352.526, 386.29, 418.42, 452.78, 485.152, 517.532] #mV
errV = Keithley(V)

muPlotter = fitPlotter("IVcharacteristic")
param = muPlotter.addGraph(I, V, errI, errV, title="I(V)")
muPlotter.drawCanvas()
muPlotter.saveCanvas("outputs/IVcharacteristic.png")

R, errR = param[1][0],param[1][1]

In [ ]:
# all measurements are in mm
import numpy as np

t = 1
err_t = 0.1
w = 10
err_w = 0.1
L = 20
err_L = 0.1

R = 64.6925
err_R = 0.0967 

rho = t * w * R / L # ohm*mm
err_rho = np.sqrt((((w*R)/L)*err_t)**2 + (((t*R)/L)*err_w)**2 + (((t*w)/L)*err_L)**2 + (((-t*w*R)/L**2)*err_R)**2) 

R_H = -131.0238 
err_R_H = 13.2014

mu = R_H / rho
err_mu = np.sqrt(((1/rho)*err_rho)**2 + ((-R_H/rho**2)*err_R_H)**2)

print(mu, err_mu)

## facoltativo: Misura del coefficiente di Hall a temperatura variabile

La misura della temperatura è effettuata mediante una misura di resistenza su un elemento
Pt100 (R$\simeq$ 100 $\ohm$). la tabella di calibrazione è su moodle.

Impostare la corrente iniettata nel campione a 8 mA e immergere il campione stesso nel
traferro, impostando un campo magnetico $\leq$ 200 mT.

A questo punto misuriamo la tensione di Hall $V_H$ (sottraendo il valore di riferimento in assenza di $B$) in funzione di $T$.

In [ ]:
RPt = [100,120,140,160,180] # fore non necessario
T_inc = np.arange(25,140,5)
T_dec = np.arange(140, 25, -5)

VH = np.arange(23) + np.random.rand(23)*3
VH_reference = np.random.rand(23)*3
print("\nT_inc: \n")
print(T_inc)
print("\nT_dec: \n")
print(T_dec)
print("\nV_H: \n")
print(VH)
print("\nV_H_reference: \n")
print(VH_reference)

In [ ]:
VH_inc_final = VH - VH_reference
VH_dec_final = VH - VH_reference

errVH = [0.3]*23
errT = [0.3]*23

In [ ]:
hallExtraIncPlotter = fitPlotter("VH_T_inc_characteristic")
VH_Tparam = hallExtraIncPlotter.addGraph(VH_inc_final, T_inc, errVH, errT, title="V_H (T) increasing")
hallExtraIncPlotter.drawCanvas()
hallExtraIncPlotter.saveCanvas("outputs/VH_T_inc_characteristic.png")

hallExtraDecPlotter = fitPlotter("VH_T_dec_characteristic")
VH_Tparam = hallExtraDecPlotter.addGraph(VH_dec_final, T_dec, errVH, errT, title="V_H (T) decreasing")
hallExtraDecPlotter.drawCanvas()
hallExtraDecPlotter.saveCanvas("outputs/VH_T_dec_characteristic.png")

### facoltativo: Magnetoresistenza